# Train mycoder on a free GPU

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run each cell from top to bottom (Shift+Enter).
3. At the end you download `mycoder.gguf` and run it offline on your own computer with Ollama.

Total time: about 20–40 minutes for the 1.5B model.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/MOHAMMEDIRFAN554/Claude.git
%cd Claude/finetune
!pip install -q -r requirements.txt

## 1. Add your own code (optional but recommended)
Upload a `.zip` of your projects in the Files panel on the left, then set `ZIP` below. Skip this to train only on the built-in examples.

In [ ]:
ZIP = ''  # e.g. '/content/my_projects.zip'
import os
if ZIP:
    !unzip -q -o "$ZIP" -d /content/mycode
    !python make_dataset.py /content/mycode --out data/train.jsonl
else:
    !mkdir -p /content/empty
    !python make_dataset.py /content/empty --out data/train.jsonl
!wc -l data/train.jsonl

## 2. Fine-tune
`Qwen2.5-Coder-1.5B-Instruct` fits the free T4. For the stronger 7B model use `--model Qwen/Qwen2.5-Coder-7B-Instruct --4bit` (slower, needs more memory).

In [ ]:
!python train_lora.py --data data/train.jsonl --model Qwen/Qwen2.5-Coder-1.5B-Instruct --epochs 3

## 3. Quick test

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = open('out/mycoder-lora/base_model.txt').read().strip()
tok = AutoTokenizer.from_pretrained('out/mycoder-lora')
m = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16, device_map='auto')
m = PeftModel.from_pretrained(m, 'out/mycoder-lora')
q = 'Find the bug:\n```python\ndef last(xs):\n    return xs[len(xs)]\n```'
ids = tok.apply_chat_template([{'role':'user','content':q}], add_generation_prompt=True, return_tensors='pt').to(m.device)
out = m.generate(ids, max_new_tokens=300, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 4. Export for Ollama and download

In [ ]:
del m; torch.cuda.empty_cache()
!python export.py --lora out/mycoder-lora
from google.colab import files
files.download('out/mycoder.gguf')
files.download('out/Modelfile')

## 5. On your own computer
Put `mycoder.gguf` and `Modelfile` in the same folder, then:
```
ollama create mycoder -f Modelfile
python coder/coder.py chat
```
No internet needed from here on.